In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent)) 

In [2]:
from src.dataset import Custom3DDataset, ObjectDataset
from src.preprocess import extract_objects, preprocess_object

data_path = "../data"
scene_dataset = Custom3DDataset(data_path)

all_objects = []

for sample in scene_dataset:
    objects = extract_objects(sample)

    for obj in objects:
        processed = preprocess_object(obj)
        all_objects.append(processed)

dataset = ObjectDataset(all_objects)

In [3]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

In [5]:
from src.model import PointNetBBox
import torch

model = PointNetBBox().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    total_loss = 0

    for batch in loader:
        points = batch["points"].cuda()   # (B, N, 3)
        center = batch["center"].cuda()
        size = batch["size"].cuda()
        yaw = batch["yaw"].cuda()

        pred = model(points)
        loss = PointNetBBox.bbox_loss(pred, (center, size, yaw))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: Loss = {total_loss:.4f}")

Epoch 0: Loss = 31.6459
Epoch 1: Loss = 21.3249
Epoch 2: Loss = 19.8916
Epoch 3: Loss = 14.7243
Epoch 4: Loss = 15.1636
Epoch 5: Loss = 14.0633
Epoch 6: Loss = 14.0344
Epoch 7: Loss = 13.4439
Epoch 8: Loss = 13.3233
Epoch 9: Loss = 13.3663
Epoch 10: Loss = 15.6475
Epoch 11: Loss = 13.2793
Epoch 12: Loss = 13.5598
Epoch 13: Loss = 14.1363
Epoch 14: Loss = 12.9477
Epoch 15: Loss = 13.4639
Epoch 16: Loss = 12.8414
Epoch 17: Loss = 13.4455
Epoch 18: Loss = 13.0829
Epoch 19: Loss = 12.3237
Epoch 20: Loss = 12.3424
Epoch 21: Loss = 12.4056
Epoch 22: Loss = 12.2641
Epoch 23: Loss = 11.9443
Epoch 24: Loss = 12.8755
Epoch 25: Loss = 11.6067
Epoch 26: Loss = 13.3750
Epoch 27: Loss = 11.6768
Epoch 28: Loss = 12.5210
Epoch 29: Loss = 11.1578
Epoch 30: Loss = 10.8179
Epoch 31: Loss = 10.6575
Epoch 32: Loss = 11.1769
Epoch 33: Loss = 9.9097
Epoch 34: Loss = 9.7626
Epoch 35: Loss = 9.4448
Epoch 36: Loss = 9.4549
Epoch 37: Loss = 10.9266
Epoch 38: Loss = 8.8163
Epoch 39: Loss = 8.7549
Epoch 40: Loss =